# EMNLP MRL 2026 — Full Experimental Pipeline
## Nepali-English Code-Switched ASR via Unconstrained Decoding

This notebook runs the finetuning and evaluation pipeline on the already code-switched dataset.
1. **CS-WER metric** implementation (the paper's novel contribution)
2. **Finetuning Whisper** (unconstrained decoding)
3. **Evaluation** (CS-WER score)
4. **Qualitative error analysis** (intra vs inter-sentential failures)

---

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer tensorboard soundfile librosa

In [ ]:
import os
import gc
import re
import csv
import json
import time
import torch
import pandas as pd
import numpy as np
import evaluate
import soundfile as sf
import librosa
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
class Config:
    # Dataset paths (update for your Kaggle environment)
    DATASET_BASE = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched"
    AUDIO_DIR = os.path.join(DATASET_BASE, "kaggle_upload", "audios_segment")

    METADATA_CSV = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle1.csv"
    GOLD_TEST_CSV = "/kaggle/working/gold_test_set.csv"

    BASE_MODEL = "openai/whisper-small"
    OUTPUT_DIR = "/kaggle/working/whisper-codeswitched-unconstrained"

    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8
    LEARNING_RATE = 1e-5
    WARMUP_STEPS = 300
    NUM_EPOCHS = 5
    EVAL_STEPS = 500
    SAVE_STEPS = 500
    LOGGING_STEPS = 50
    FP16 = True

    SAMPLING_RATE = 16000
    TEST_SIZE = 0.05
    SEED = 42

config = Config()
print("Config loaded.")

---
## Section 1: CS-WER Metric Implementation

This is the paper's **key novel metric**. Standard WER treats all words equally. CS-WER isolates the error rate strictly to English loanwords.

Approach:
1. Use `jiwer.process_words()` for word-level alignment
2. Classify each reference token as English (Latin script) or Nepali (Devanagari)
3. Compute substitution/deletion/insertion rates over English tokens only

In [ ]:
from jiwer import process_words

def is_english_word(word):
    cleaned = re.sub(r'[^\w]', '', word)
    if not cleaned:
        return False
    first_char = cleaned[0]
    return first_char.isascii() and first_char.isalpha()

def compute_cs_wer(references, hypotheses):
    total_eng, correct_eng, sub_eng, del_eng, ins_eng = 0, 0, 0, 0, 0
    total_nep, correct_nep, sub_nep, del_nep = 0, 0, 0, 0

    wer_metric = evaluate.load("wer")
    overall_wer = wer_metric.compute(predictions=hypotheses, references=references)

    for ref, hyp in zip(references, hypotheses):
        output = process_words(ref, hyp)

        for chunk in output.alignments[0]:
            ref_slice = output.references[0][chunk.ref_start_idx:chunk.ref_end_idx]
            hyp_slice = output.hypotheses[0][chunk.hyp_start_idx:chunk.hyp_end_idx]

            if chunk.type == "equal":
                for w in ref_slice:
                    if is_english_word(w):
                        total_eng += 1; correct_eng += 1
                    else:
                        total_nep += 1; correct_nep += 1
            elif chunk.type == "substitute":
                for w in ref_slice:
                    if is_english_word(w):
                        total_eng += 1; sub_eng += 1
                    else:
                        total_nep += 1; sub_nep += 1
            elif chunk.type == "delete":
                for w in ref_slice:
                    if is_english_word(w):
                        total_eng += 1; del_eng += 1
                    else:
                        total_nep += 1; del_nep += 1
            elif chunk.type == "insert":
                for w in hyp_slice:
                    if is_english_word(w):
                        ins_eng += 1

    cs_wer = (sub_eng + del_eng + ins_eng) / total_eng * 100 if total_eng > 0 else 0.0
    nep_wer = (sub_nep + del_nep) / total_nep * 100 if total_nep > 0 else 0.0

    return {
        "overall_wer": round(overall_wer * 100, 2),
        "cs_wer": round(cs_wer, 2),
        "nepali_wer": round(nep_wer, 2),
        "english_total": total_eng,
    }

---
## Section 2: Data Loading & Preprocessing

In [ ]:
def load_metadata(csv_path):
    df = pd.read_csv(csv_path, engine="python", on_bad_lines="warn", quoting=csv.QUOTE_MINIMAL)

    def resolve_audio_path(rel):
        return os.path.join(config.AUDIO_DIR, os.path.basename(str(rel)))

    df["audio_path"] = df["path"].apply(resolve_audio_path)
    missing = ~df["audio_path"].apply(os.path.exists)
    if missing.sum() > 0:
        print(f"  Removing {missing.sum()} missing audio files")
        df = df[~missing].reset_index(drop=True)

    df["transcription"] = df["transcription"].astype(str)
    df = df.dropna(subset=["transcription"])
    df = df[df["transcription"].str.strip() != ""].reset_index(drop=True)
    df["transcription"] = df["transcription"].str.replace(r"\s+", " ", regex=True).str.strip()
    return df

def build_hf_dataset(df, feature_extractor, tokenizer):
    dataset = Dataset.from_dict({
        "audio": df["audio_path"].tolist(),
        "transcription": df["transcription"].tolist(),
    }).cast_column("audio", Audio(sampling_rate=config.SAMPLING_RATE))

    dataset = dataset.train_test_split(test_size=config.TEST_SIZE, seed=config.SEED)

    def prepare(batch):
        audio = batch["audio"]
        batch["input_features"] = feature_extractor(
            audio["array"], sampling_rate=audio["sampling_rate"]
        ).input_features[0]
        batch["labels"] = tokenizer(batch["transcription"]).input_ids
        return batch

    dataset = dataset.map(prepare, remove_columns=dataset.column_names["train"], num_proc=1)
    return dataset

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

---
## Section 3: Model Finetuning

In [ ]:
wer_metric = evaluate.load("wer")

def get_compute_metrics_fn(tokenizer):
    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = tokenizer.pad_token_id
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": wer}
    return compute_metrics

def train_model(dataset, output_dir):
    feature_extractor = WhisperFeatureExtractor.from_pretrained(config.BASE_MODEL)
    tokenizer = WhisperTokenizer.from_pretrained(config.BASE_MODEL, language="ne", task="transcribe")
    processor = WhisperProcessor.from_pretrained(config.BASE_MODEL, language="ne", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(config.BASE_MODEL)

    model.generation_config.task = "transcribe"
    model.generation_config.max_new_tokens = 225
    model.generation_config.no_repeat_ngram_size = 3

    # Unconstrained decoding for code-switching
    model.generation_config.language = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    print("  Decoder: UNCONSTRAINED (code-switching mode)")

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=config.BATCH_SIZE,
        per_device_eval_batch_size=config.BATCH_SIZE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,
        learning_rate=config.LEARNING_RATE,
        warmup_steps=config.WARMUP_STEPS,
        lr_scheduler_type="linear",
        num_train_epochs=config.NUM_EPOCHS,
        eval_steps=config.EVAL_STEPS,
        save_steps=config.SAVE_STEPS,
        logging_steps=config.LOGGING_STEPS,
        eval_strategy="steps",
        predict_with_generate=True,
        generation_max_length=225,
        fp16=config.FP16,
        dataloader_num_workers=2,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        report_to=["tensorboard"],
        push_to_hub=False,
        remove_unused_columns=False,
        label_names=["labels"],
        seed=config.SEED,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        data_collator=data_collator,
        compute_metrics=get_compute_metrics_fn(tokenizer),
        processing_class=processor.feature_extractor,
    )

    torch.cuda.empty_cache()
    gc.collect()
    trainer.train()

    final_path = os.path.join(output_dir, "final-model")
    trainer.save_model(final_path)
    processor.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)

    print(f"Model saved to {final_path}")
    return final_path

---
## Section 4: Inference & Error Analysis

In [ ]:
def run_inference(model_path, audio_paths):
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_path)
    tokenizer = WhisperTokenizer.from_pretrained(model_path, language="ne", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(model_path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    model.generation_config.language = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    model.generation_config.max_new_tokens = 225
    model.generation_config.no_repeat_ngram_size = 3

    predictions = []
    for path in audio_paths:
        audio, sr = librosa.load(path, sr=config.SAMPLING_RATE)
        inputs = feature_extractor(audio, sampling_rate=sr, return_tensors="pt").to(model.device)
        with torch.no_grad():
            ids = model.generate(inputs.input_features)
        predictions.append(tokenizer.decode(ids[0], skip_special_tokens=True))

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return predictions

def error_analysis(references, hypotheses, n_examples=20):
    categories = {"intra_sentential": [], "inter_sentential": [], "mixed_morphology": [], "proper_noun": []}
    for ref, hyp in zip(references, hypotheses):
        output = process_words(ref, hyp)
        for chunk in output.alignments[0]:
            if chunk.type in ("substitute", "delete"):
                ref_words = output.references[0][chunk.ref_start_idx:chunk.ref_end_idx]
                hyp_words = output.hypotheses[0][chunk.hyp_start_idx:chunk.hyp_end_idx]
                eng_words = [w for w in ref_words if is_english_word(w)]
                if not eng_words: continue

                example = {"reference": " ".join(ref_words), "hypothesis": " ".join(hyp_words) if hyp_words else "[DELETED]"}
                has_suffix = any(re.search(r'[a-zA-Z]+[\u0900-\u097F]', w) for w in eng_words)
                if has_suffix: categories["mixed_morphology"].append(example)
                elif len(eng_words) >= 2: categories["inter_sentential"].append(example)
                elif any(w[0].isupper() for w in eng_words if w): categories["proper_noun"].append(example)
                else: categories["intra_sentential"].append(example)

    print("\n" + "=" * 80)
    print("ERROR ANALYSIS")
    print("=" * 80)
    for cat, examples in categories.items():
        print(f"\n--- {cat.upper()} ({len(examples)} errors) ---")
        for ex in examples[:n_examples]:
            print(f'  REF: {ex["reference"]} \n  HYP: {ex["hypothesis"]}\n')
    return categories

---
## Section 5: Run the Experiments
Execute each cell below in order.

### Step 1: Load datasets and preprocess

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(config.BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(config.BASE_MODEL, language="ne", task="transcribe")

df_metadata = load_metadata(config.METADATA_CSV)
dataset = build_hf_dataset(df_metadata, feature_extractor, tokenizer)
print(f"Dataset: {len(dataset['train'])} train / {len(dataset['test'])} val")

# Save a portion of the data as the gold test set for evaluation
test_size = int(len(df_metadata) * config.TEST_SIZE)
gold_df = df_metadata.iloc[-test_size:].copy()
gold_df.to_csv(config.GOLD_TEST_CSV, index=False)
print(f"Saved {len(gold_df)} items to {config.GOLD_TEST_CSV} for evaluation.")

### Step 2: Finetune the model (Unconstrained Decoder)

In [ ]:
trained_model_path = train_model(dataset, config.OUTPUT_DIR)

### Step 3: Run Inference & Evaluation

In [ ]:
gold_df = pd.read_csv(config.GOLD_TEST_CSV)
audio_paths = gold_df["audio_path"].tolist()
references = gold_df["transcription"].tolist()

predictions = run_inference(trained_model_path, audio_paths)
metrics = compute_cs_wer(references, predictions)

print("\nRESULTS:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

### Step 4: Error analysis

In [ ]:
categories = error_analysis(references, predictions)